# 01 - Chunking strategies

## Question

What happens if we use the same word window for every file instead of following each document's structure? This notebook compares:

1. fixed word windows with overlap
2. semantic grouping of neighboring paragraphs
3. document-aware chunks (sections for Markdown, one row per table chunk, one record per JSON chunk, pages/paragraphs for PDF and text)
4. child chunks that expand to their parent after retrieval

In [1]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
core_notebook_path = ROOT / "notebooks/_rag_demo_core.ipynb"
get_ipython().run_line_magic("run", f'"{core_notebook_path}"')
DATA = ROOT / "data"
ARTIFACTS = ROOT / "artifacts"
print(f"Project root: {ROOT}")
openai_settings = get_openai_settings(require_api_key=False)
print(f"OpenAI configured: {openai_settings.configured}; "
      f"embedding={openai_settings.embedding_model}; chat={openai_settings.chat_model}")


Project root: /Users/snomula/Downloads/sripal/Context Engineern-RAG
OpenAI configured: False; embedding=text-embedding-3-small; chat=gpt-5-mini


In [2]:
generate_pdf(DATA / "raw/northstar_safety_bulletin.pdf")

documents = load_corpus(DATA / "raw")
fixed = chunk_corpus(documents, "fixed", size=80, overlap=15)
semantic = chunk_corpus(documents, "semantic", threshold=0.10, max_words=140)
aware = chunk_corpus(documents, "document_aware", max_words=110, overlap=15)
print({"fixed": len(fixed), "semantic": len(semantic), "document_aware": len(aware)})


{'fixed': 24, 'semantic': 18, 'document_aware': 43}


In [3]:
from collections import Counter
for name, chunks in [("fixed", fixed), ("semantic", semantic), ("aware", aware)]:
    sizes = [len(chunk.text.split()) for chunk in chunks]
    print(name, {"count": len(sizes), "min": min(sizes),
                 "mean": round(sum(sizes)/len(sizes), 1), "max": max(sizes),
                 "types": Counter(c.metadata.get("element_type", "flat") for c in chunks)})


fixed {'count': 24, 'min': 17, 'mean': 68.6, 'max': 80, 'types': Counter({'flat': 24})}
semantic {'count': 18, 'min': 7, 'mean': 74.6, 'max': 139, 'types': Counter({'flat': 18})}
aware {'count': 43, 'min': 10, 'mean': 32.8, 'max': 106, 'types': Counter({'table_row': 16, 'section': 12, 'paragraph': 10, 'json_record': 3, 'page': 2})}


### Table check: do the headers survive?

A good table chunk repeats the section and column names on every row, so a retrieved row makes sense by itself instead of being a lone number.

In [4]:
table_chunks = [c for c in aware if c.doc_id == "manual-orion-x2"
                and c.metadata.get("element_type") == "table_row"]
for chunk in table_chunks:
    print(f"\n{chunk.chunk_id} | heading={chunk.metadata.get('heading')}")
    print(chunk.text)



manual-orion-x2:aware:002:00 | heading=Torque specifications
Torque specifications | Component: Front axle nut | Torque: 42 N·m | Thread treatment: Medium-strength threadlocker | Inspection interval: Every 1,000 km

manual-orion-x2:aware:003:00 | heading=Torque specifications
Torque specifications | Component: Rear axle nut | Torque: 55 N·m | Thread treatment: None | Inspection interval: Every 1,000 km

manual-orion-x2:aware:004:00 | heading=Torque specifications
Torque specifications | Component: Brake caliper bolt | Torque: 24 N·m | Thread treatment: Medium-strength threadlocker | Inspection interval: Every 500 km

manual-orion-x2:aware:005:00 | heading=Torque specifications
Torque specifications | Component: Handlebar clamp bolt | Torque: 8 N·m | Thread treatment: None | Inspection interval: Every 500 km

manual-orion-x2:aware:006:00 | heading=Diagnostic codes
Diagnostic codes | Code: E17 | Meaning: Motor temperature sensor signal out of range | First action: Inspect connector J4 f

### Parent-child retrieval

We index small child chunks because they match more precisely. Once a child wins, its `parent_id` lets us pull in the full section/row/page it came from. The index doesn't fill up with big diluted chunks, but the model still gets enough surrounding context.

In [5]:
parents = build_parent_lookup(documents)
sample = next(c for c in aware if c.doc_id == "handbook-2026" and "Receipts" in c.text)
print("CHILD:\n", sample.text)
print("\nPARENT ID:", sample.parent_id)
print("\nEXPANDED PARENT:\n", parents[sample.parent_id])


CHILD:
 Section: Travel and expenses
Expense reports are due within **15 calendar days** after a trip ends. Receipts are mandatory for any single expense of **$50 or more**. Managers may approve domestic hotel rates up to $275 per night; higher rates require Finance approval.

Flights under six scheduled hours must be booked in economy. Premium economy is allowed for flights from six through nine hours. Business class requires a scheduled flight longer than nine hours and vice-president approval.

PARENT ID: handbook-2026:parent:003

EXPANDED PARENT:
 Expense reports are due within **15 calendar days** after a trip ends. Receipts are mandatory for any single expense of **$50 or more**. Managers may approve domestic hotel rates up to $275 per night; higher rates require Finance approval.

Flights under six scheduled hours must be booked in economy. Premium economy is allowed for flights from six through nine hours. Business class requires a scheduled flight longer than nine hours and vi

## Chunking trade-offs

| Strategy | Good at | Usually fails on |
|---|---|---|
| Fixed + overlap | simple, predictable | cuts tables/sections, duplicates text |
| Recursive | keeps paragraphs/sentences | doesn't know document semantics |
| Semantic | follows topic changes | sensitive to threshold/model |
| Document-aware | keeps rows, records, headings, pages | more work per file type |
| Parent-child | precise match plus surrounding context | needs parent mapping and budgets |

**Don't tune chunk size on its own.** It depends on the embedding model, the kinds of questions, reranking, the context window and the eval labels.